# Bulk RNA-seq Time-Course Analysis Template

本 notebook 用于处理时间梯度 RNA-seq 数据，包括分组均值聚合、Mfuzz 软聚类、趋势可视化、热图，以及各时间点的差异表达分析。

## 1. Parameter Configuration

In [ ]:
EXPR_FILE <- "./1-DEG/vsd_matrix.csv"     # genes x samples; exported by RNAseq_General
META_FILE <- "./1-DEG/colData.csv"         # must contain sample, condition, and time columns
GENE_COLUMN <- NULL                         # NULL = use rownames/first column
SAMPLE_COLUMN <- "sample"
TIME_COLUMN <- "time"                      # numeric or factor time column
GROUP_COLUMN <- "condition"               # optional treatment/group column
TIME_LEVELS <- NULL                         # e.g. c("Day0", "Day7", "Day14", "Day21")

# Time-course clustering
RUN_MFUZZ <- TRUE
MFUZZ_N_CLUSTERS <- 5
MFUZZ_MIN_ACORE <- 0.7
MFUZZ_SEED <- 2025

# Raw count input required for time-point vs baseline DEG
RAW_COUNTS_FILE <- "./0-Data/raw_counts.tsv"    # genes x samples, raw integer counts
COUNT_META_FILE <- "./0-Data/metadata.csv"       # must contain sample, time, condition; optional subject
COUNT_GENE_COL <- "gene_name"
COUNT_SAMPLE_COL <- "sample"
COUNT_BIOTYPE_COL <- NULL
COUNT_BIOTYPE_FILTER <- "protein_coding"

# Optional paired design (repeated measures)
SUBJECT_COL <- NULL                            # e.g. "patient_id"; NULL = unpaired

# Pairwise DEG: each time point vs baseline
RUN_TIMEPOINT_DEG <- TRUE
BASELINE_TIME <- NULL                          # NULL = earliest TIME_LEVELS; or specify e.g. "Day0"
DEG_PADJ_CUTOFF <- 0.05
DEG_LFC_CUTOFF <- 0.5
MIN_COUNT <- 10

# Output
OUTDIR <- "RNAseq_TimeCourse_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("Mfuzz", "ComplexHeatmap", "circlize", "clusterProfiler", "org.Hs.eg.db", "org.Mm.eg.db", "DESeq2"))

suppressPackageStartupMessages({
  library(tidyverse)
  library(ComplexHeatmap)
  library(circlize)
  library(Mfuzz)
  library(clusterProfiler)
  library(DESeq2)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "io_utils.R"))
source(file.path(LIB_DIR, "data_utils.R"))
source(file.path(LIB_DIR, "deg_utils.R"))
source(file.path(LIB_DIR, "enrichment_utils.R"))
source(file.path(LIB_DIR, "timecourse_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")

## 3. Load Expression and Metadata

In [ ]:
# Load expression matrix and metadata with validation
expr <- read_expression_matrix(EXPR_FILE, gene_column = GENE_COLUMN)

required_meta_cols <- c(SAMPLE_COLUMN, TIME_COLUMN)
if (!is.null(GROUP_COLUMN)) required_meta_cols <- c(required_meta_cols, GROUP_COLUMN)
meta <- read_metadata(
  META_FILE,
  sample_column = SAMPLE_COLUMN,
  required_columns = required_meta_cols,
  time_column = TIME_COLUMN,
  time_levels = TIME_LEVELS,
  group_column = GROUP_COLUMN
)
if (is.null(TIME_LEVELS)) TIME_LEVELS <- levels(meta[[TIME_COLUMN]])

validate_samples_match(colnames(expr), meta[[SAMPLE_COLUMN]], strict_order = TRUE)
expr <- expr[, meta[[SAMPLE_COLUMN]], drop = FALSE]

# Mfuzz/clustering requires normalized expression derived from raw counts.
scale_info <- validate_expression_contract(expr, expected = "vst")

cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")
print(table(meta[[TIME_COLUMN]], useNA = "ifany"))

## 4. Aggregate Expression by Time Point

In [ ]:
expr_mean <- aggregate_expr_by_group(expr, meta[[TIME_COLUMN]])
expr_mean <- expr_mean[, TIME_LEVELS, drop = FALSE]  # ensure order
write.csv(expr_mean, file.path(OUTDIR, "mean_expression_by_time.csv"))
cat("Aggregated expression:", nrow(expr_mean), "genes x", ncol(expr_mean), "time points\n")


## 5. Mfuzz Time-Course Soft Clustering

In [ ]:
if (RUN_MFUZZ) {
  eset <- prepare_mfuzz_eset(expr_mean)
  mfuzz_result <- run_mfuzz(eset, n_clusters = MFUZZ_N_CLUSTERS, seed = MFUZZ_SEED)
  cluster_df <- extract_mfuzz_clusters(mfuzz_result, eset = eset, min_acore = MFUZZ_MIN_ACORE)
  write_mfuzz_cluster_table(cluster_df, file.path(OUTDIR, "mfuzz_clusters.csv"))

  cat("Mfuzz cluster sizes:\n")
  print(summarize_mfuzz_clusters(cluster_df))

  # Trend plots
  plot_mfuzz_trends_pdf(
    eset, mfuzz_result,
    filename = file.path(OUTDIR, "mfuzz_trends.pdf"),
    time_labels = TIME_LEVELS,
    width = mm_to_in(183), height = 6.4
  )

  # Heatmap of core genes by cluster
  core_df <- cluster_df[cluster_df$core_gene, ]
  if (nrow(core_df) >= 2) {
    group_colors <- make_group_colors(TIME_LEVELS)
    plot_timecourse_heatmap_pdf(
      expr, core_df,
      group_vec = meta[[TIME_COLUMN]],
      group_levels = TIME_LEVELS,
      group_colors = group_colors,
      filename = file.path(OUTDIR, "mfuzz_core_heatmap.pdf"),
      width = mm_to_in(183), height = mm_to_in(247)
    )
  }

  # ORA per cluster
  universe <- map_symbols_to_entrez(rownames(expr), org.Hs.eg.db)$ENTREZID
  ora_results <- run_mfuzz_cluster_ora(cluster_df, org_db = org.Hs.eg.db, universe = universe)
  for (cl_name in names(ora_results)) {
    prefix <- file.path(OUTDIR, paste0("GO_ORA_", cl_name))
    write.csv(as.data.frame(ora_results[[cl_name]]), paste0(prefix, ".csv"), row.names = FALSE)
    plot_enrich_suite_pdf(ora_results[[cl_name]], prefix, cl_name)
  }
}


## 6. Time-Point vs Baseline DEG (Optional)

In [ ]:
if (RUN_TIMEPOINT_DEG) {
  if (!file.exists(RAW_COUNTS_FILE)) {
    stop("RAW_COUNTS_FILE not found: ", RAW_COUNTS_FILE,
         "\nProvide raw integer counts with columns for gene_name and samples.")
  }

  rawcount <- read_count_table(RAW_COUNTS_FILE, "tsv")
  if (!is.null(COUNT_BIOTYPE_COL) && COUNT_BIOTYPE_COL %in% colnames(rawcount)) {
    rawcount <- rawcount[rawcount[[COUNT_BIOTYPE_COL]] == COUNT_BIOTYPE_FILTER, ]
  }
  count_col_names <- detect_count_columns(rawcount, COUNT_GENE_COL, NULL)
  count_meta <- read.csv(COUNT_META_FILE, check.names = FALSE)
  count_samples <- intersect(count_col_names, count_meta[[COUNT_SAMPLE_COL]])
  if (length(count_samples) == 0) {
    stop("No matching samples between raw counts and count metadata.")
  }
  rawcount <- rawcount[, c(COUNT_GENE_COL, count_samples), drop = FALSE]
  count_meta <- count_meta[match(count_samples, count_meta[[COUNT_SAMPLE_COL]]), ]

  countData <- build_count_matrix(
    rawcount, COUNT_GENE_COL, count_samples, count_meta[[COUNT_SAMPLE_COL]],
    duplicate_report_file = file.path(OUTDIR, "1-DEG_Timepoint", "Duplicated_gene_symbols.csv")
  )
  countData <- filter_low_count_genes(countData, count_meta[[TIME_COLUMN]], MIN_COUNT)$count_data

  col_data <- data.frame(
    sample = count_meta[[COUNT_SAMPLE_COL]],
    stringsAsFactors = FALSE
  )
  col_data[[TIME_COLUMN]] <- count_meta[[TIME_COLUMN]]
  if (!is.null(GROUP_COLUMN) && GROUP_COLUMN %in% colnames(count_meta)) {
    col_data[[GROUP_COLUMN]] <- count_meta[[GROUP_COLUMN]]
  }
  if (!is.null(SUBJECT_COL) && SUBJECT_COL %in% colnames(count_meta)) {
    col_data[[SUBJECT_COL]] <- count_meta[[SUBJECT_COL]]
  }
  rownames(col_data) <- col_data$sample

  if (is.null(BASELINE_TIME)) {
    baseline_time <- TIME_LEVELS[1]
  } else {
    baseline_time <- BASELINE_TIME
  }

  cat("Running time-point vs baseline DEG with baseline:", baseline_time, "\n")
  tp_res_list <- run_timepoint_vs_baseline_deseq2(
    count_data = countData,
    col_data = col_data,
    time_col = TIME_COLUMN,
    baseline_time = baseline_time,
    condition_col = if (!is.null(GROUP_COLUMN) && GROUP_COLUMN %in% colnames(col_data)) GROUP_COLUMN else NULL,
    subject_col = SUBJECT_COL,
    alpha = max(DEG_PADJ_CUTOFF, 0.05)
  )

  tp_deg_dir <- file.path(OUTDIR, "1-DEG_Timepoint")
  tp_summary <- write_timepoint_deg_results(
    tp_res_list,
    outdir = tp_deg_dir,
    pvalue_column = "padj",
    lfc_column = "log2FoldChange_shrunken"
  )

  plot_timepoint_deg_summary_pdf(
    tp_summary,
    filename = file.path(OUTDIR, "3-Visualization", "Timepoint_DEG_summary.pdf")
  )

  for (comp_name in names(tp_res_list)) {
    plot_volcano_pdf(
      tp_res_list[[comp_name]],
      comp_name = comp_name,
      pvalue_thresh = DEG_PADJ_CUTOFF,
      log2fc_thresh = DEG_LFC_CUTOFF,
      pvalue_column = "padj",
      lfc_column = "log2FoldChange_shrunken",
      filename = file.path(OUTDIR, "3-Visualization", paste0("Volcano_", comp_name, ".pdf"))
    )
  }

  cat("Time-point vs baseline DEG complete. Results saved to", tp_deg_dir, "\n")
} else {
  cat("Time-point DEG skipped (RUN_TIMEPOINT_DEG = FALSE).\n")
}


### 5.1 Publication-Grade Theme Dot-heatmap

Group Mfuzz cluster ORA terms into biological themes for a manuscript-ready overview.

In [ ]:
if (RUN_MFUZZ) {
  theme_outdir <- file.path(OUTDIR, "ThemeEnrichment")
  dir.create(theme_outdir, showWarnings = FALSE, recursive = TRUE)
  theme_defs <- default_enrichment_themes()

  drop_empty <- function(lst) lst[sapply(lst, function(x) !is.null(x) && nrow(as.data.frame(x)) > 0)]
  ora_map <- drop_empty(ora_results)

  if (length(ora_map) > 0) {
    p <- plot_theme_dotheatmap_from_results(
      ora_map,
      filename = file.path(theme_outdir, "Theme_dotheatmap_GO_ORA_mfuzz_clusters.pdf"),
      title = "GO ORA Biological Themes (Mfuzz Clusters)",
      subtitle = "GO-BP ORA | top terms per theme per cluster",
      theme_defs = theme_defs,
      ontology_filter = "BP",
      top_n = 6
    )
    if (!is.null(p)) print(p)
  }
}


## 7. Save Session

In [ ]:
save.image(file = file.path(OUTDIR, "timecourse_workspace.Rdata"))
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
cat("Analysis complete. Outputs saved to", OUTDIR, "\n")
